# ScamShield - encoder fine-tuning on a free Colab T4

Reproduces the committed numbers for the two fine-tuned multilingual encoders.
The same script runs locally on Apple silicon (`python -m src.models.transformer`), which is
how the numbers in the README were produced. This notebook exists so the result can be
reproduced on a free GPU without a Mac.

Models compared: `ai4bharat/IndicBERTv2-MLM-only` (Indic-focused encoder) and `xlm-roberta-base`.
MuRIL is the encoder named in the project brief, but its Hugging Face repositories are gated and
need an access token; `configs/muril.json` is not provided because the run could not be verified
here. Add your token, point `--model` at `google/muril-base`, and the same code path applies.

In [ ]:
!git clone https://github.com/pavan-dangeti/ScamShield-AI.git
%cd ScamShield-AI
!pip install -q -r requirements-dev.txt torch transformers peft accelerate

In [ ]:
import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Build the corpus

Same two commands CI runs. `build_dataset` writes `data/processed/{train,val}.jsonl`.

In [ ]:
!python -m scripts.download_data
!python -m scripts.build_dataset
!python -m scripts.check_leakage --threshold 0.90

## Train

About 40 minutes per model on a free T4. Change the model id and config to switch encoders.

In [ ]:
!python -m src.models.transformer --model ai4bharat/IndicBERTv2-MLM-only --config configs/indicbertv2.json

In [ ]:
!python -m src.models.transformer --model xlm-roberta-base --config configs/xlmr.json

## Evaluate every model with the same harness

In [ ]:
!python -m scripts.evaluate_models --models tfidf-lr indicbertv2-mlm xlmr-base
print(open('results/model_comparison.md').read())

## LoRA fine-tune of a small open model (optional)

Same verdict format as the prompted LLM, so the two LLM arms stay comparable.

In [ ]:
!python -m src.models.lora --model Qwen/Qwen2.5-0.5B-Instruct --config configs/lora_qwen05b.json